In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
from pathlib import Path
import glob
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils import data
from torch.utils.data import Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
# M4 데이터 로드
# 입력: WSI tile image paths + tile 좌표 + selected RNA-seq feature + basic clinical(age, sex) + OS label
# Clinical 입력에서는 pathologic_stage/T/N/M/tumor_grade를 사용하지 않습니다.

DATA_PATH = Path("../../data")
RESULT_PATH = Path("../../results")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"
RNASEQ_SELECTED_NAME = "top_literature_guided_survival_cox_1500"
RNASEQ_DIM = 1500
RNASEQ_PATH = DST_PATH / "RNAseq_selected" / RNASEQ_SELECTED_NAME

M4_OUTPUT_PATH = RESULT_PATH / "pancreatic_cancer_pathology" / "M4"
M4_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = ["TCGA_PAAD", "CPTAC_PDAC"]
DATASET_NAME = "TCGA_CPTAC"
PATCH_INPUT_SIZE = 512
PRELOAD_RESIZED_TILES = True  # dataset 구성 단계에서 512x512 tile을 CPU memory에 preload합니다.
PRELOAD_TILE_SIZE = PATCH_INPUT_SIZE
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)
TEST_SIZE = 0.2
VALID_SIZE = 0.25  # train_valid 내부 비율. 전체 기준 0.8 * 0.25 = 0.2
REQUIRE_COMPLETE_24M_HORIZONS = True

MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]
CLINICAL_FEATURE_COLUMNS = ["age_years_z", "sex_male", "sex_female"]


def load_clinical_json(dataset: str, case_id: str) -> dict:
    clinical_json_path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not clinical_json_path.exists():
        return {}
    with open(clinical_json_path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    return clinical_json.get("clinical", {})


def get_rnaseq_feature_path(dataset: str, case_id: str) -> Path:
    return RNASEQ_PATH / dataset / f"{case_id}_rnaseq_top_literature_guided_survival_cox_{RNASEQ_DIM}.npy"


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask

    os_time_days = float(os_time_days)
    os_event = int(os_event)
    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def get_patch_padding(image_size: int = PATCH_INPUT_SIZE) -> tuple[int, int, int, int]:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    target_size = int(np.ceil(image_size / patch_size) * patch_size)
    pad_total = max(0, target_size - image_size)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return (pad_left, pad_left, pad_right, pad_right)


def get_model_input_size(image_size: int = PATCH_INPUT_SIZE) -> int:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    return int(np.ceil(image_size / patch_size) * patch_size)


def get_train_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_train_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    # dataset 구성 단계에서 이미 512x512로 resize된 image에 대해 padding/augmentation/normalization만 수행합니다.
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def estimate_tile_cache_gb(n_tiles: int, image_size: int = PRELOAD_TILE_SIZE) -> float:
    return n_tiles * image_size * image_size * 3 / (1024 ** 3)


def preload_resized_tile_images(tile_index: pd.DataFrame, image_size: int = PRELOAD_TILE_SIZE) -> dict[str, np.ndarray]:
    unique_paths = sorted(tile_index["tile_path"].astype(str).unique())
    expected_gb = estimate_tile_cache_gb(len(unique_paths), image_size=image_size)
    print(f"Preloading resized tiles: {len(unique_paths):,} tiles, {image_size}x{image_size}, expected uint8 memory ~{expected_gb:.2f} GB")

    cache = {}
    resize_size = (image_size, image_size)
    for tile_path in tqdm(unique_paths, desc="Preload resized tile images", unit="tile"):
        with Image.open(tile_path) as image:
            image = image.convert("RGB")
            image = image.resize(resize_size, resample=Image.BILINEAR)
            cache[tile_path] = np.asarray(image, dtype=np.uint8).copy()
    return cache


def add_tile_coordinates(tile_df: pd.DataFrame) -> pd.DataFrame:
    tile_df = tile_df.copy()
    if {"x_level0", "y_level0"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_level0"].astype(int)
        tile_df["y"] = tile_df["y_level0"].astype(int)
    elif {"x_total_matrix", "y_total_matrix"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_total_matrix"].astype(int)
        tile_df["y"] = tile_df["y_total_matrix"].astype(int)
    else:
        raise ValueError("tile metadata must contain x/y coordinate columns.")
    return tile_df


def get_slide_dimensions(tile_df: pd.DataFrame) -> tuple[int, int]:
    source_size = tile_df["source_tile_size_px"].astype(float)
    width = int((tile_df["x"].astype(float) + source_size).max())
    height = int((tile_df["y"].astype(float) + source_size).max())
    return width, height


def load_case_tiles(dataset: str, metadata_path: Path) -> tuple[pd.DataFrame, dict]:
    case_id = metadata_path.parent.name
    case_dir = metadata_path.parent
    tile_df = pd.read_csv(metadata_path)

    tile_df = add_tile_coordinates(tile_df)
    slide_width, slide_height = get_slide_dimensions(tile_df)

    tile_df["tile_path"] = tile_df["tile_path"].astype(str)
    tile_df = tile_df[tile_df["tile_path"].map(lambda x: Path(x).exists())].copy()
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height

    source_size = tile_df["source_tile_size_px"].astype(float)
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height

    clinical = load_clinical_json(dataset, case_id)
    os_time = clinical.get("os_time_days", tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan)
    os_event = clinical.get("os_event", tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan)
    age_years = clinical.get("age_years", np.nan)
    sex = str(clinical.get("sex", "unknown")).lower()

    rnaseq_path = get_rnaseq_feature_path(dataset, case_id)
    y, mask = make_horizon_label_mask(os_time, os_event)
    case_record = {
        "dataset": dataset,
        "slide_uid": f"{dataset}::{case_id}",
        "case_id": case_id,
        "case_dir": case_dir.as_posix(),
        "metadata_path": metadata_path.as_posix(),
        "n_tiles": int(len(tile_df)),
        "slide_width": slide_width,
        "slide_height": slide_height,
        "os_time_days": float(os_time) if pd.notna(os_time) else np.nan,
        "os_event": int(os_event) if pd.notna(os_event) else np.nan,
        "age_years": float(age_years) if pd.notna(age_years) else np.nan,
        "sex": sex,
        "known_horizon_count": int(mask.sum()),
        "rnaseq_path": rnaseq_path.as_posix(),
        "has_rnaseq_selected": rnaseq_path.exists(),
    }
    for i, name in enumerate(HORIZON_NAMES):
        case_record[name] = float(y[i])
        case_record[f"mask_{name}"] = float(mask[i])
    tile_df["dataset"] = dataset
    tile_df["case_id"] = case_id
    tile_df["slide_uid"] = case_record["slide_uid"]
    return tile_df, case_record


case_records = []
tile_index_list = []
for dataset in DATASET_NAMES:
    metadata_paths = sorted((IMAGE_PATH / dataset).glob("*/tile_metadata.csv"))
    for metadata_path in tqdm(metadata_paths, desc=f"Load {dataset} metadata"):
        tile_df, case_record = load_case_tiles(dataset, metadata_path)
        if case_record["n_tiles"] > 0:
            tile_index_list.append(tile_df)
        case_records.append(case_record)

all_slide_df = pd.DataFrame(case_records)
tile_index_df = pd.concat(tile_index_list, ignore_index=True)


complete_horizon_mask = all_slide_df[[f"mask_{name}" for name in HORIZON_NAMES]].eq(1).all(axis=1)
required_horizon_mask = complete_horizon_mask if REQUIRE_COMPLETE_24M_HORIZONS else all_slide_df["known_horizon_count"].gt(0)

eligible_mask = (
    all_slide_df["os_time_days"].notna()
    & all_slide_df["os_event"].notna()
    & required_horizon_mask
    & all_slide_df["n_tiles"].gt(0)
    & all_slide_df["has_rnaseq_selected"].eq(True)
    & all_slide_df["age_years"].notna()
    & all_slide_df["sex"].isin(["male", "female"])
)
excluded_df = all_slide_df[~eligible_mask].copy()
if not excluded_df.empty:
    excluded_df["exclude_reason"] = np.select(
        [
            excluded_df["n_tiles"].le(0),
            excluded_df["os_time_days"].isna(),
            excluded_df["os_event"].isna(),
            ~required_horizon_mask.loc[excluded_df.index],
            excluded_df["age_years"].isna(),
            ~excluded_df["sex"].isin(["male", "female"]),
            ~excluded_df["has_rnaseq_selected"].eq(True),
        ],
        ["no_tiles", "missing_os_time", "missing_os_event", "incomplete_required_horizon", "missing_age", "missing_sex", "missing_rnaseq_selected"],
        default="unknown",
    )

slide_df = all_slide_df[eligible_mask].copy()
slide_df["os_event"] = slide_df["os_event"].astype(int)
tile_index_df = tile_index_df[tile_index_df["slide_uid"].isin(slide_df["slide_uid"])].copy()

slide_df["stratify_group"] = slide_df["dataset"].astype(str) + "_event" + slide_df["os_event"].astype(str)
stratify_for_test = slide_df["stratify_group"] if slide_df["stratify_group"].value_counts().min() >= 2 else slide_df["os_event"]
train_valid_df, test_df = train_test_split(slide_df, test_size=TEST_SIZE, random_state=SEED, stratify=stratify_for_test)
stratify_for_valid = train_valid_df["stratify_group"] if train_valid_df["stratify_group"].value_counts().min() >= 2 else train_valid_df["os_event"]
train_df, valid_df = train_test_split(train_valid_df, test_size=VALID_SIZE, random_state=SEED, stratify=stratify_for_valid)

age_mean = float(train_df["age_years"].mean())
age_std = float(train_df["age_years"].std(ddof=0))
age_std = age_std if age_std > 0 else 1.0
clinical_stats = {"age_mean": age_mean, "age_std": age_std, "sex_encoding": ["sex_male", "sex_female"]}


def add_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["age_years_z"] = (df["age_years"].astype(float) - age_mean) / age_std
    df["sex_male"] = df["sex"].eq("male").astype(float)
    df["sex_female"] = df["sex"].eq("female").astype(float)
    return df

train_df = add_clinical_features(train_df)
valid_df = add_clinical_features(valid_df)
test_df = add_clinical_features(test_df)
slide_df = pd.concat([train_df, valid_df, test_df], axis=0).sort_index()

slide_df.to_csv(M4_OUTPUT_PATH / "m4_tcga_cptac_horizon_rnaseq_clinical_slide_manifest.csv", index=False)
tile_index_df.to_csv(M4_OUTPUT_PATH / "m4_tcga_cptac_horizon_tile_index.csv", index=False)
excluded_df.to_csv(M4_OUTPUT_PATH / "m4_tcga_cptac_horizon_excluded_cases.csv", index=False)
with open(M4_OUTPUT_PATH / "m4_clinical_stats.json", "w", encoding="utf-8") as f:
    json.dump(clinical_stats, f, indent=2, ensure_ascii=False)

print("all_slide_df:", all_slide_df.shape)
print("complete 24m horizon cases:", int(complete_horizon_mask.sum()))
print("slide_df:", slide_df.shape)
print("excluded_df:", excluded_df.shape)
print("tile_index_df:", tile_index_df.shape)
print("clinical stats:", clinical_stats)

if PRELOAD_RESIZED_TILES:
    if "TILE_IMAGE_CACHE" not in globals() or not TILE_IMAGE_CACHE:
        TILE_IMAGE_CACHE = preload_resized_tile_images(tile_index_df, image_size=PRELOAD_TILE_SIZE)
    else:
        print(f"Using existing TILE_IMAGE_CACHE: {len(TILE_IMAGE_CACHE):,} tiles")
else:
    TILE_IMAGE_CACHE = {}

print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("cached tiles:", len(TILE_IMAGE_CACHE))

display(slide_df[["case_id", "age_years", "sex"] + CLINICAL_FEATURE_COLUMNS].head())

horizon_summary = []
for name in HORIZON_NAMES:
    mask_col = f"mask_{name}"
    known = slide_df[mask_col].eq(1)
    horizon_summary.append({
        "horizon": name,
        "known_n": int(known.sum()),
        "dead_n": int(slide_df.loc[known, name].sum()),
        "alive_n": int(known.sum() - slide_df.loc[known, name].sum()),
        "unknown_n": int((~known).sum()),
        "dead_rate_known": float(slide_df.loc[known, name].mean()) if known.any() else np.nan,
    })
display(pd.DataFrame(horizon_summary))


class M4SlideDataset(Dataset):
    """Case-level pathology + RNA-seq + basic clinical dataset for Cox MIL training."""

    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            slide_uid: group.sort_values(["y", "x"]).reset_index(drop=True)
            for slide_uid, group in tile_index.groupby("slide_uid")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["slide_uid"]]
        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        label = row[HORIZON_NAMES].to_numpy(np.float32)
        clinical_features = row[CLINICAL_FEATURE_COLUMNS].to_numpy(np.float32)
        slide_size = np.array([row["slide_width"], row["slide_height"]], dtype=np.float32)
        rnaseq_features = np.load(row["rnaseq_path"]).astype(np.float32)
        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "slide_uid": row["slide_uid"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "slide_size": torch.from_numpy(slide_size),
            "clinical_features": torch.from_numpy(clinical_features),
            "rnaseq_features": torch.from_numpy(rnaseq_features),
            "horizon_months": torch.tensor(HORIZON_MONTHS, dtype=torch.float32),
            "label": torch.from_numpy(label),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
        }


class M4TileDataset(Dataset):
    def __init__(self, tile_index: pd.DataFrame, transform=None, tile_cache: dict[str, np.ndarray] | None = None):
        self.tile_index = tile_index.reset_index(drop=True).copy()
        self.transform = transform
        self.tile_cache = tile_cache or {}

    def __len__(self):
        return len(self.tile_index)

    def __getitem__(self, idx):
        row = self.tile_index.iloc[idx]
        tile_path = row["tile_path"]
        if tile_path in self.tile_cache:
            image = Image.fromarray(self.tile_cache[tile_path])
        else:
            with Image.open(tile_path) as image_file:
                image = image_file.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coords = row[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {"image": image, "coords": torch.from_numpy(coords), "case_id": row["case_id"],
            "slide_uid": row["slide_uid"], "tile_path": tile_path}


train_case_ids = set(train_df["slide_uid"])
valid_case_ids = set(valid_df["slide_uid"])
test_case_ids = set(test_df["slide_uid"])
assert len(train_case_ids & valid_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(valid_case_ids & test_case_ids) == 0

train_dataset = M4SlideDataset(train_df, tile_index_df)
valid_dataset = M4SlideDataset(valid_df, tile_index_df)
test_dataset = M4SlideDataset(test_df, tile_index_df)

tile_train_transform = get_train_cached_patch_transform() if TILE_IMAGE_CACHE else get_train_patch_transform()
tile_eval_transform = get_eval_cached_patch_transform() if TILE_IMAGE_CACHE else get_eval_patch_transform()

train_tile_dataset = M4TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(train_case_ids)],
    transform=tile_train_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
valid_tile_dataset = M4TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(valid_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
test_tile_dataset = M4TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(test_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)

split_df = slide_df[["dataset", "slide_uid", "case_id", "os_time_days", "os_event", "age_years", "sex", "rnaseq_path"] + CLINICAL_FEATURE_COLUMNS + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].copy()
split_df["split"] = "unused"
split_df.loc[split_df["slide_uid"].isin(train_case_ids), "split"] = "train"
split_df.loc[split_df["slide_uid"].isin(valid_case_ids), "split"] = "valid"
split_df.loc[split_df["slide_uid"].isin(test_case_ids), "split"] = "test"
split_df.to_csv(M4_OUTPUT_PATH / "m4_tcga_cptac_horizon_rnaseq_clinical_case_splits.csv", index=False)

print("slide splits:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("tile splits:", len(train_tile_dataset), len(valid_tile_dataset), len(test_tile_dataset))
print("split x dataset")
display(pd.crosstab(split_df["split"], split_df["dataset"]))
print("split x os_event")
display(pd.crosstab(split_df["split"], split_df["os_event"]))
print("split x dataset x os_event")
display(pd.crosstab([split_df["split"], split_df["dataset"]], split_df["os_event"]))

sample = train_dataset[0]
print("sample case:", sample["case_id"])
print("n_tiles:", len(sample["tile_paths"]))
print("clinical_features:", sample["clinical_features"])
print("rnaseq_features:", sample["rnaseq_features"].shape)
print("label:", sample["label"])
print("model input size:", get_model_input_size(PATCH_INPUT_SIZE), "padding:", get_patch_padding(PATCH_INPUT_SIZE))


In [ ]:
# M4 모델 정의 및 UNI/UNI2-h 로드, loss/optimizer 구성
# 구조: frozen UNI/UNI2-h feature extractor + risk_topk MIL + RNA-seq embedding + clinical embedding

import torch
from torch import nn
import timm
from huggingface_hub import hf_hub_download

from scripts.models.discrete_survival import (
    cox_ph_loss,
    harrell_c_index,
)

from scripts.models.m4_pathology_rnaseq_clinical_mil import (
    M4ModelConfig,
    PathologyRNASeqClinicalMIL,
    sample_tiles,
    build_optimizer,
)

print("device:", device)

MAX_TILES_PER_SLIDE = 512
FEATURE_BATCH_SIZE = 64
SPATIAL_DIM = 128
CLINICAL_EMBED_DIM = 16
RNASEQ_HIDDEN_DIM = 256
RNASEQ_EMBED_DIM = 64
RNASEQ_DROPOUT = 0.50
FUSION_DIM = 128
MIL_HIDDEN_DIM = 64
USE_SPATIAL_EMBEDDING = False
POOLING_MODE = "risk_topk"
DROPOUT = 0.50
N_OUTPUTS = 1  # patient-level Cox risk score
CLINICAL_DIM = len(CLINICAL_FEATURE_COLUMNS)
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-3

FEATURE_EXTRACTOR_NAME = "UNI2-h"
UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}


def load_uni_feature_extractor(name: str = FEATURE_EXTRACTOR_NAME, device: torch.device = device) -> tuple[nn.Module, int]:
    cfg = UNI_BACKBONES[name]
    print(f"Loading {name}: {cfg['repo_id']} / {cfg['timm_kwargs']['model_name']}")
    model = timm.create_model(pretrained=False, **cfg["timm_kwargs"])
    weight_path = hf_hub_download(repo_id=cfg["repo_id"], filename=cfg["filename"], local_files_only=False)
    state_dict = torch.load(weight_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print("missing:", len(missing), "unexpected:", len(unexpected))
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device)
    feature_dim = int(getattr(model, "num_features", cfg["feature_dim"]))
    print(f"{name} loaded. feature_dim={feature_dim}")
    return model, feature_dim


feature_extractor, M4_FEATURE_DIM = load_uni_feature_extractor(name=FEATURE_EXTRACTOR_NAME, device=device)
FEATURE_EXTRACTOR_PATCH_SIZE = int(UNI_BACKBONES[FEATURE_EXTRACTOR_NAME]["timm_kwargs"]["patch_size"])
print("FEATURE_EXTRACTOR_PATCH_SIZE:", FEATURE_EXTRACTOR_PATCH_SIZE)
print("PATCH_INPUT_SIZE:", PATCH_INPUT_SIZE, "-> model input size:", get_model_input_size(PATCH_INPUT_SIZE))
print("PATCH_PADDING:", get_patch_padding(PATCH_INPUT_SIZE))

# 2번 셀에서 만든 tile-level dataset transform은 feature extractor patch size를 알기 전 생성되므로 여기서 갱신합니다.
# resized tile cache가 있으면 512 resize를 반복하지 않고 padding/augmentation/normalization만 수행합니다.
if bool(globals().get("TILE_IMAGE_CACHE", {})):
    train_tile_dataset.transform = get_train_cached_patch_transform()
    valid_tile_dataset.transform = get_eval_cached_patch_transform()
    test_tile_dataset.transform = get_eval_cached_patch_transform()
else:
    train_tile_dataset.transform = get_train_patch_transform()
    valid_tile_dataset.transform = get_eval_patch_transform()
    test_tile_dataset.transform = get_eval_patch_transform()

m4_config = M4ModelConfig(
    feature_dim=M4_FEATURE_DIM,
    coord_dim=6,
    clinical_dim=CLINICAL_DIM,
    rnaseq_dim=RNASEQ_DIM,
    spatial_dim=SPATIAL_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    rnaseq_hidden_dim=RNASEQ_HIDDEN_DIM,
    rnaseq_embed_dim=RNASEQ_EMBED_DIM,
    fusion_dim=FUSION_DIM,
    mil_hidden_dim=MIL_HIDDEN_DIM,
    n_outputs=N_OUTPUTS,
    dropout=DROPOUT,
    rnaseq_dropout=RNASEQ_DROPOUT,
    max_tiles=MAX_TILES_PER_SLIDE,
    feature_batch_size=FEATURE_BATCH_SIZE,
    freeze_feature_extractor=True,
    use_spatial_embedding=USE_SPATIAL_EMBEDDING,
    pooling_mode=POOLING_MODE,
)


def m4_loss_fn(logits: torch.Tensor, os_time_days: torch.Tensor, os_event: torch.Tensor) -> torch.Tensor:
    return cox_ph_loss(
        risk_scores=logits.float().reshape(-1),
        times=os_time_days.float().reshape(-1),
        events=os_event.float().reshape(-1),
    )

def initialize_m4_model(feature_extractor: nn.Module, feature_dim: int = M4_FEATURE_DIM) -> tuple[PathologyRNASeqClinicalMIL, torch.optim.Optimizer]:
    config = M4ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        clinical_dim=CLINICAL_DIM,
        rnaseq_dim=RNASEQ_DIM,
        spatial_dim=SPATIAL_DIM,
        clinical_embed_dim=CLINICAL_EMBED_DIM,
        rnaseq_hidden_dim=RNASEQ_HIDDEN_DIM,
        rnaseq_embed_dim=RNASEQ_EMBED_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        rnaseq_dropout=RNASEQ_DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
        use_spatial_embedding=USE_SPATIAL_EMBEDDING,
        pooling_mode=POOLING_MODE,
    )
    model = PathologyRNASeqClinicalMIL(feature_extractor=feature_extractor, config=config).to(device)
    optimizer = build_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    return model, optimizer


model, optimizer = initialize_m4_model(feature_extractor, feature_dim=M4_FEATURE_DIM)
print("M4 model initialized.")
print("FEATURE_EXTRACTOR_NAME:", FEATURE_EXTRACTOR_NAME)
print("M4_FEATURE_DIM:", M4_FEATURE_DIM)
print("CLINICAL_DIM:", CLINICAL_DIM, CLINICAL_FEATURE_COLUMNS)
print("RNASEQ_DIM:", RNASEQ_DIM)
print("RNASEQ_EMBED_DIM:", RNASEQ_EMBED_DIM)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("MAX_TILES_PER_SLIDE:", MAX_TILES_PER_SLIDE)
print("FEATURE_BATCH_SIZE:", FEATURE_BATCH_SIZE)
print("USE_SPATIAL_EMBEDDING:", USE_SPATIAL_EMBEDDING)
print("POOLING_MODE:", POOLING_MODE)
print("N_OUTPUTS:", N_OUTPUTS, "patient-level risk score")


In [ ]:
# M4 학습 하이퍼파라미터 정의

MODEL_PATH = Path("../../model")
M4_MODEL_DIR = MODEL_PATH / "pancreatic_cancer_pathology" / "M4" / FEATURE_EXTRACTOR_NAME
M4_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
PATIENCE = 15
MIN_DELTA = 1e-4
MONITOR_METRIC = "valid_c_index"
MONITOR_MODE = "max"
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
SLIDE_BATCH_SIZE = 1
LOG_EVERY_EPOCHS = 10
SAVE_EVERY_EPOCHS = 10
CASE_BATCH_SIZE = 16

BEST_CHECKPOINT_PATH = M4_MODEL_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = M4_MODEL_DIR / "last_checkpoint.pt"
TRAIN_LOG_PATH = M4_MODEL_DIR / "training_log.csv"
CONFIG_PATH = M4_MODEL_DIR / "training_config.json"

training_config = {
    "model": "M4_pathology_rnaseq_basic_clinical",
    "dataset": DATASET_NAME,
    "feature_extractor": FEATURE_EXTRACTOR_NAME,
    "clinical_features": CLINICAL_FEATURE_COLUMNS,
    "rnaseq_selection_name": RNASEQ_SELECTED_NAME,
    "rnaseq_dim": RNASEQ_DIM,
    "rnaseq_hidden_dim": RNASEQ_HIDDEN_DIM,
    "rnaseq_embed_dim": RNASEQ_EMBED_DIM,
    "rnaseq_dropout": RNASEQ_DROPOUT,
    "clinical_stats": clinical_stats,
    "patch_input_size": PATCH_INPUT_SIZE,
    "model_input_size": get_model_input_size(PATCH_INPUT_SIZE),
    "preload_resized_tiles": PRELOAD_RESIZED_TILES,
    "preload_tile_size": PRELOAD_TILE_SIZE,
    "patch_padding": get_patch_padding(PATCH_INPUT_SIZE),
    "feature_extractor_patch_size": FEATURE_EXTRACTOR_PATCH_SIZE,
    "effective_mpp": 1.0 * 1024 / PATCH_INPUT_SIZE,
    "max_tiles_per_slide": MAX_TILES_PER_SLIDE,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "horizon_months": HORIZON_MONTHS,
    "horizon_names": HORIZON_NAMES,
    "loss_function": "cox_partial_likelihood",
    "output_interpretation": "patient_level_risk_score",
    "case_batch_size": CASE_BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "monitor_metric": MONITOR_METRIC,
    "monitor_mode": MONITOR_MODE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "spatial_dim": SPATIAL_DIM,
    "clinical_embed_dim": CLINICAL_EMBED_DIM,
    "fusion_dim": FUSION_DIM,
    "mil_hidden_dim": MIL_HIDDEN_DIM,
    "use_spatial_embedding": USE_SPATIAL_EMBEDDING,
    "pooling_mode": POOLING_MODE,
    "use_amp": USE_AMP,
    "amp_dtype": str(AMP_DTYPE),
    "seed": SEED,
    "model_dir": M4_MODEL_DIR.as_posix(),
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=MONITOR_MODE, factor=0.5, patience=5, min_lr=1e-6)

print("M4_MODEL_DIR:", M4_MODEL_DIR)
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("EPOCHS:", EPOCHS)
print("PATIENCE:", PATIENCE)
print("CASE_BATCH_SIZE:", CASE_BATCH_SIZE)
print("USE_AMP:", USE_AMP)
print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("scheduler:", scheduler.__class__.__name__)
print("MONITOR_METRIC:", MONITOR_METRIC, MONITOR_MODE)


In [ ]:
# M4 학습 및 validation
# patient-level Cox partial likelihood loss, C-index, KM curve, log-rank, hazard ratio, checkpoint, scheduler

from PIL import Image
import math
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def load_tile_tensor_batch(tile_paths: list[str], transform, device: torch.device) -> torch.Tensor:
    images = []
    use_cache = bool(globals().get("TILE_IMAGE_CACHE", {}))
    for path in tile_paths:
        if use_cache and path in TILE_IMAGE_CACHE:
            image = Image.fromarray(TILE_IMAGE_CACHE[path])
        else:
            with Image.open(path) as image_file:
                image = image_file.convert("RGB")
        images.append(transform(image))
    return torch.stack(images, dim=0).to(device, non_blocking=True)


def prepare_slide_batch(sample: dict, training: bool) -> dict:
    if bool(globals().get("TILE_IMAGE_CACHE", {})):
        transform = get_train_cached_patch_transform() if training else get_eval_cached_patch_transform()
    else:
        transform = get_train_patch_transform() if training else get_eval_patch_transform()
    selected_paths, selected_coords, selected_indices = sample_tiles(
        sample["tile_paths"],
        sample["coords"],
        max_tiles=MAX_TILES_PER_SLIDE,
        training=training,
    )
    tile_images = load_tile_tensor_batch(selected_paths, transform=transform, device=device)
    return {
        "tile_images": tile_images,
        "coords": selected_coords.to(device, non_blocking=True),
        "rnaseq_features": sample["rnaseq_features"].unsqueeze(0).to(device, non_blocking=True).float(),
        "clinical_features": sample["clinical_features"].unsqueeze(0).to(device, non_blocking=True).float(),
        "os_time_days": sample["os_time_days"].reshape(1).to(device, non_blocking=True).float(),
        "os_event": sample["os_event"].reshape(1).to(device, non_blocking=True).long(),
        "dataset": sample["dataset"],
        "case_id": sample["case_id"],
        "slide_uid": sample.get("slide_uid", sample["case_id"]),
        "n_tiles": len(selected_paths),
    }


def logrank_test(times: np.ndarray, events: np.ndarray, group: np.ndarray) -> dict[str, float]:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    group = np.asarray(group, dtype=int)
    event_times = np.sort(np.unique(times[(events == 1) & np.isfinite(times)]))
    observed_high = 0.0
    expected_high = 0.0
    variance_high = 0.0
    for t in event_times:
        at_risk = times >= t
        event_at_t = (times == t) & (events == 1)
        n = float(at_risk.sum())
        if n <= 1:
            continue
        n_high = float((at_risk & (group == 1)).sum())
        d = float(event_at_t.sum())
        d_high = float((event_at_t & (group == 1)).sum())
        expected = d * n_high / n
        var = n_high * (n - n_high) * d * (n - d) / (n * n * max(n - 1.0, 1.0))
        observed_high += d_high
        expected_high += expected
        variance_high += var
    chi2_stat = (observed_high - expected_high) ** 2 / max(variance_high, 1e-12)
    p_value = float(math.erfc(math.sqrt(max(chi2_stat, 0.0) / 2.0)))
    return {
        "observed_high": observed_high,
        "expected_high": expected_high,
        "chi2": chi2_stat,
        "p_value": p_value,
    }


def cox_binary_hr(times: np.ndarray, events: np.ndarray, group: np.ndarray, max_iter: int = 50) -> dict[str, float]:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=float)
    x = np.asarray(group, dtype=float)
    valid = np.isfinite(times)
    times, events, x = times[valid], events[valid], x[valid]
    if len(np.unique(x)) < 2 or events.sum() == 0:
        return {"hr": float("nan"), "hr_ci_low": float("nan"), "hr_ci_high": float("nan"), "beta": float("nan"), "se": float("nan")}
    beta = 0.0
    info = np.nan
    for _ in range(max_iter):
        score = 0.0
        info = 0.0
        for t, e, xi in zip(times, events, x):
            if e != 1:
                continue
            risk = times >= t
            w = np.exp(np.clip(beta * x[risk], -50, 50))
            sw = w.sum()
            mean_x = (w * x[risk]).sum() / sw
            mean_x2 = (w * x[risk] * x[risk]).sum() / sw
            score += xi - mean_x
            info += mean_x2 - mean_x * mean_x
        if info <= 1e-12:
            break
        step = score / info
        beta += step
        if abs(step) < 1e-6:
            break
    se = float(np.sqrt(1.0 / max(info, 1e-12)))
    hr = float(np.exp(beta))
    return {
        "hr": hr,
        "hr_ci_low": float(np.exp(beta - 1.96 * se)),
        "hr_ci_high": float(np.exp(beta + 1.96 * se)),
        "beta": float(beta),
        "se": se,
    }


def km_curve(times: np.ndarray, events: np.ndarray) -> pd.DataFrame:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    order = np.argsort(times)
    times, events = times[order], events[order]
    surv = 1.0
    rows = [{"time": 0.0, "survival": 1.0}]
    for t in np.unique(times[events == 1]):
        at_risk = np.sum(times >= t)
        deaths = np.sum((times == t) & (events == 1))
        if at_risk > 0:
            surv *= 1.0 - deaths / at_risk
            rows.append({"time": float(t), "survival": float(surv)})
    return pd.DataFrame(rows)


def compute_epoch_metrics(risk_scores: list[float], times: list[float], events: list[int], datasets: list[str], case_ids: list[str]) -> dict:
    risk_array = np.asarray(risk_scores, dtype=float)
    time_array = np.asarray(times, dtype=float)
    event_array = np.asarray(events, dtype=int)
    metrics = {
        "c_index": harrell_c_index(time_array, event_array, risk_array) if len(risk_array) else float("nan"),
        "risk_mean": float(np.mean(risk_array)) if len(risk_array) else float("nan"),
        "risk_std": float(np.std(risk_array)) if len(risk_array) else float("nan"),
    }
    if len(risk_array) and np.isfinite(risk_array).all():
        cutoff = float(np.median(risk_array))
        group = (risk_array >= cutoff).astype(int)
        lr = logrank_test(time_array, event_array, group)
        hr = cox_binary_hr(time_array, event_array, group)
        metrics.update({
            "risk_cutoff_median": cutoff,
            "logrank_p_value": lr["p_value"],
            "logrank_chi2": lr["chi2"],
            "hr_high_vs_low": hr["hr"],
            "hr_ci_low": hr["hr_ci_low"],
            "hr_ci_high": hr["hr_ci_high"],
        })
    else:
        metrics.update({"risk_cutoff_median": float("nan"), "logrank_p_value": float("nan"), "logrank_chi2": float("nan"), "hr_high_vs_low": float("nan"), "hr_ci_low": float("nan"), "hr_ci_high": float("nan")})
    metrics["prediction_df"] = pd.DataFrame({
        "dataset": datasets,
        "case_id": case_ids,
        "os_time_days": time_array,
        "os_event": event_array,
        "risk_score": risk_array,
    })
    if len(risk_array):
        metrics["prediction_df"]["risk_group"] = np.where(risk_array >= metrics["risk_cutoff_median"], "High risk", "Low risk")
    return metrics


def metrics_for_checkpoint(metrics: dict) -> dict:
    return {k: v for k, v in metrics.items() if k != "prediction_df"}


def finalize_case_batch(logits_list, times_list, events_list, training: bool) -> torch.Tensor:
    logits_tensor = torch.cat(logits_list, dim=0)
    times_tensor = torch.cat(times_list, dim=0)
    events_tensor = torch.cat(events_list, dim=0)
    loss = m4_loss_fn(logits_tensor, times_tensor, events_tensor)
    if training:
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
    return loss


def run_one_epoch(dataset, training: bool, epoch: int) -> dict:
    model.train(training)
    total_loss = 0.0
    total_batches = 0
    all_risks = []
    all_times = []
    all_events = []
    all_datasets = []
    all_case_ids = []
    logits_list = []
    times_list = []
    events_list = []

    progress = tqdm(range(len(dataset)), desc=f"Epoch {epoch:03d} {'train' if training else 'valid'}", leave=False)
    for idx in progress:
        sample = dataset[idx]
        with torch.set_grad_enabled(training):
            batch = prepare_slide_batch(sample, training=training)
            outputs = model(batch["tile_images"], batch["coords"], batch["rnaseq_features"], batch["clinical_features"])
            risk = outputs["logits"].reshape(-1)
            logits_list.append(risk)
            times_list.append(batch["os_time_days"])
            events_list.append(batch["os_event"].float())

        all_risks.append(float(risk.detach().cpu().item()))
        all_times.append(float(batch["os_time_days"].detach().cpu().item()))
        all_events.append(int(batch["os_event"].detach().cpu().item()))
        all_datasets.append(batch["dataset"])
        all_case_ids.append(batch["case_id"])

        is_batch_ready = len(logits_list) >= CASE_BATCH_SIZE or idx == len(dataset) - 1
        if is_batch_ready:
            loss = finalize_case_batch(logits_list, times_list, events_list, training=training)
            total_loss += float(loss.detach().cpu())
            total_batches += 1
            logits_list, times_list, events_list = [], [], []
            running_loss = total_loss / max(total_batches, 1)
            running_metrics = compute_epoch_metrics(all_risks, all_times, all_events, all_datasets, all_case_ids)
            progress.set_postfix({
                "avg_loss": f"{running_loss:.4f}",
                "c_index": "nan" if np.isnan(running_metrics["c_index"]) else f"{running_metrics['c_index']:.3f}",
                "hr": "nan" if np.isnan(running_metrics["hr_high_vs_low"]) else f"{running_metrics['hr_high_vs_low']:.2f}",
                "p": "nan" if np.isnan(running_metrics["logrank_p_value"]) else f"{running_metrics['logrank_p_value']:.3g}",
            })
            del loss

        del batch, outputs, risk
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metrics = {"loss": total_loss / max(total_batches, 1)}
    metrics.update(compute_epoch_metrics(all_risks, all_times, all_events, all_datasets, all_case_ids))
    return metrics


def get_monitor_score(metrics: dict, prefix: str = "valid") -> float:
    key = MONITOR_METRIC.replace(f"{prefix}_", "")
    return float(metrics.get(key, np.nan))


def save_checkpoint(path: Path, epoch: int, metrics: dict, is_best: bool) -> None:
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": {"train": metrics_for_checkpoint(metrics["train"]), "valid": metrics_for_checkpoint(metrics["valid"]), "monitor_score": metrics["monitor_score"]},
        "training_config": training_config,
        "is_best": is_best,
    }, path)


def plot_km_by_risk(pred_df: pd.DataFrame, title: str) -> None:
    if pred_df.empty or pred_df["risk_group"].nunique() < 2:
        return
    plt.figure(figsize=(6, 5))
    for group_name, part in pred_df.groupby("risk_group"):
        km = km_curve(part["os_time_days"].to_numpy(float), part["os_event"].to_numpy(int))
        plt.step(km["time"] / 30.4375, km["survival"], where="post", label=f"{group_name} (n={len(part)})")
    plt.title(title)
    plt.xlabel("Months")
    plt.ylabel("Overall survival probability")
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


def plot_training_history(history: list[dict], valid_pred_df: pd.DataFrame | None = None) -> None:
    hist_df = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0, 0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
    axes[0, 0].plot(hist_df["epoch"], hist_df["valid_loss"], label="valid")
    axes[0, 0].set_title("Cox Partial Likelihood Loss")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].legend()

    axes[0, 1].plot(hist_df["epoch"], hist_df["train_c_index"], label="train")
    axes[0, 1].plot(hist_df["epoch"], hist_df["valid_c_index"], label="valid")
    axes[0, 1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
    axes[0, 1].set_title("C-index")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].legend()

    axes[1, 0].plot(hist_df["epoch"], hist_df["valid_hr_high_vs_low"], label="valid HR")
    axes[1, 0].set_title("Valid HR: high vs low risk")
    axes[1, 0].set_xlabel("epoch")
    axes[1, 0].legend()

    axes[1, 1].plot(hist_df["epoch"], hist_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("epoch")
    axes[1, 1].set_yscale("log")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()
    display(hist_df.tail(10))
    if valid_pred_df is not None:
        last = hist_df.iloc[-1]
        plot_km_by_risk(
            valid_pred_df,
            title=f"Validation KM by median risk | epoch={int(last['epoch'])}, C-index={last['valid_c_index']:.3f}, p={last['valid_logrank_p']:.3g}",
        )


history = []
best_score = -np.inf if MONITOR_MODE == "max" else np.inf
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch(train_dataset, training=True, epoch=epoch)
    valid_metrics = run_one_epoch(valid_dataset, training=False, epoch=epoch)

    monitor_score = get_monitor_score(valid_metrics, prefix="valid")
    scheduler.step(monitor_score)
    current_lr = optimizer.param_groups[0]["lr"]

    improved = monitor_score > best_score + MIN_DELTA if MONITOR_MODE == "max" else monitor_score < best_score - MIN_DELTA
    if improved:
        best_score = monitor_score
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "valid_loss": valid_metrics["loss"],
        "train_c_index": train_metrics["c_index"],
        "valid_c_index": valid_metrics["c_index"],
        "train_logrank_p": train_metrics["logrank_p_value"],
        "valid_logrank_p": valid_metrics["logrank_p_value"],
        "train_hr_high_vs_low": train_metrics["hr_high_vs_low"],
        "train_hr_ci_low": train_metrics["hr_ci_low"],
        "train_hr_ci_high": train_metrics["hr_ci_high"],
        "valid_hr_high_vs_low": valid_metrics["hr_high_vs_low"],
        "valid_hr_ci_low": valid_metrics["hr_ci_low"],
        "valid_hr_ci_high": valid_metrics["hr_ci_high"],
        "train_risk_mean": train_metrics["risk_mean"],
        "valid_risk_mean": valid_metrics["risk_mean"],
        "train_risk_std": train_metrics["risk_std"],
        "valid_risk_std": valid_metrics["risk_std"],
        "lr": current_lr,
    }
    history.append(row)

    log_df = pd.DataFrame(history)
    log_df.to_csv(TRAIN_LOG_PATH, index=False)

    checkpoint_metrics = {"train": train_metrics, "valid": valid_metrics, "monitor_score": monitor_score}
    save_checkpoint(LAST_CHECKPOINT_PATH, epoch, checkpoint_metrics, is_best=False)
    if improved:
        save_checkpoint(BEST_CHECKPOINT_PATH, epoch, checkpoint_metrics, is_best=True)

    status = "best saved" if improved else f"no improve {epochs_without_improvement}/{PATIENCE}"
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['loss']:.4f} valid_loss={valid_metrics['loss']:.4f} | "
        f"train_c={train_metrics['c_index']:.3f} valid_c={valid_metrics['c_index']:.3f} | "
        f"valid_HR={valid_metrics['hr_high_vs_low']:.2f} "
        f"(95% CI {valid_metrics['hr_ci_low']:.2f}-{valid_metrics['hr_ci_high']:.2f}) | "
        f"logrank_p={valid_metrics['logrank_p_value']:.3g} | "
        f"lr={current_lr:.2e} | best_epoch={best_epoch} ({status})"
    )

    if epoch % LOG_EVERY_EPOCHS == 0 or improved:
        plot_training_history(history, valid_metrics.get("prediction_df"))

    if epoch % SAVE_EVERY_EPOCHS == 0:
        save_checkpoint(M4_MODEL_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, checkpoint_metrics, is_best=False)

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch={best_epoch}, best {MONITOR_METRIC}={best_score:.4f}")
        break

print("training finished")
print("best_epoch:", best_epoch)
print("best_score:", best_score)
print("best checkpoint:", BEST_CHECKPOINT_PATH)
